In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
dbutils.widgets.text("start_date", "", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "", "End Date (YYYY-MM-DD)")
dbutils.widgets.text("tipo_carga", "", "Tipo de carga")

# Parámetros de fechas (strings)
start_date_str = dbutils.widgets.get("start_date")
end_date_str = dbutils.widgets.get("end_date")
tipo_carga = dbutils.widgets.get("tipo_carga")

In [0]:
start_date_str ,  end_date_str

In [0]:
# Crear esquema silver si no existe
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.silver.hd_lima_air_pollution (
    fecha_ts TIMESTAMP COMMENT 'Fecha and tiempo, Unix UTC',
    id_ubigeo STRING COMMENT 'Código único de distrito (UBIGEO)',
    nom_distrito STRING COMMENT 'Nombre del distrito',
    nom_provincia STRING COMMENT 'Nombre de la provincia',
    nom_departamento STRING COMMENT 'Nombre del departamento',
    latitud DOUBLE COMMENT 'Latitud del distrito',
    longitud DOUBLE COMMENT 'Longitud del distrito',
    index_air_quality INT COMMENT 'Air Quality Index (1=Good, 5=Very Poor)',
    conc_co DOUBLE COMMENT 'Concentration of CO (Carbon monoxide), μg/m3',
    conc_no DOUBLE COMMENT 'Concentration of NO (Nitrogen monoxide), μg/m3',
    conc_no2 DOUBLE COMMENT 'Concentration of NO2 (Nitrogen dioxide), μg/m3',
    conc_o3 DOUBLE COMMENT 'Concentration of O3 (Ozone), μg/m3',
    conc_so2 DOUBLE COMMENT 'Concentration of SO2 (Sulphur dioxide), μg/m3',
    conc_pm2_5 DOUBLE COMMENT 'Concentration of PM2.5 (Fine particles matter), μg/m3',
    conc_pm10 DOUBLE COMMENT 'Concentration of PM10 (Coarse particulate matter), μg/m3',
    conc_nh3 DOUBLE COMMENT 'Concentration of NH3 (Ammonia), μg/m3',
    anio INT COMMENT 'Año derivado de dt',
    mes INT COMMENT 'Mes derivado de dt',
    dia INT COMMENT 'Día derivado de dt',
    fecha DATE COMMENT 'Fecha derivada de dt',
    fecha_data DATE COMMENT 'Fecha en que se actualiza la informacion de la tabla'
) 
USING DELTA
PARTITIONED BY (fecha);

In [0]:
# -------------------------------
# 2. Función de escritura flexible
# -------------------------------
def write_silver_air_pollution(df_silver, mode="incremental", table_name="silver.hd_lima_air_pollution"):
    """
    Escribe el DataFrame en la tabla Silver de contaminación del aire.
    
    Parámetros:
    - df_silver: DataFrame transformado con columnas ya listas (incluyendo 'fecha').
    - mode: "total" para sobrescribir toda la tabla, "incremental" para sobrescribir solo las particiones afectadas.
    - table_name: nombre de la tabla destino en Silver.
    """
    
    writer = df_silver.write.format("delta")#.option("overwriteSchema", "true")
    
    if mode == "total":
        # Sobrescribir toda la tabla
        writer.mode("overwrite").partitionBy("fecha").saveAsTable(table_name)
        print(f"Tabla {table_name} sobrescrita completamente.")
    
    elif mode == "incremental":
        # Detectar fechas presentes en el lote
        #fechas_lote = [row["fecha"] for row in df_silver.select("fecha").distinct().collect()]
        #condicion = " OR ".join([f"fecha = '{f}'" for f in fechas_lote])
        
        # Sobrescribir solo las particiones afectadas
        print(f"Escritura en fechas desde {start_date_str} hasta {end_date_str}")
        
        writer.mode("overwrite") \
        .option(
            "replaceWhere",
            f"fecha >= DATE'{start_date_str}' AND fecha <= DATE'{end_date_str}'"
        ) \
        .saveAsTable(table_name)

        print(f"Tabla {table_name} sobrescrita solo en fechas desde {start_date_str} hasta {end_date_str}")
    
    else:
        raise ValueError("El parámetro 'mode' debe ser 'total' o 'incremental'.")


In [0]:
# Definir esquemas para parsear los JSON internos
main_schema = StructType([
    StructField("aqi", StringType(), True)
])

components_schema = StructType([
    StructField("co", StringType(), True),
    StructField("no", StringType(), True),
    StructField("no2", StringType(), True),
    StructField("o3", StringType(), True),
    StructField("so2", StringType(), True),
    StructField("pm2_5", StringType(), True),
    StructField("pm10", StringType(), True),
    StructField("nh3", StringType(), True)
])

# Leer Bronze
if tipo_carga == "incremental":
    df_bronze = spark.table("workspace.bronze.hist_air_pollution").filter((col("fecha") >= start_date_str) & (col("fecha") <= end_date_str)).select(
        col("dt"), col("main"), col("components"),
        col("ubigeo"),col("distrito"),col("provincia"),col("departamento"),col("latitud"), col("longitud")
    )

elif tipo_carga == "total":
    df_bronze = spark.table("workspace.bronze.hist_air_pollution").select(
        col("dt"), col("main"), col("components"),
        col("ubigeo"),col("distrito"),col("provincia"),col("departamento"),col("latitud"), col("longitud")
    )

# Parsear JSON y aplicar transformaciones
df_silver = (
    df_bronze
    .withColumn("main_parsed", from_json(col("main"), main_schema))
    .withColumn("components_parsed", from_json(col("components"), components_schema))
    # Renombrar y castear
    .withColumn("index_air_quality", col("main_parsed.aqi").cast("int"))
    .withColumn("conc_co", round(col("components_parsed.co"),2).cast("double"))
    .withColumn("conc_no", round(col("components_parsed.no"),2).cast("double"))
    .withColumn("conc_no2", round(col("components_parsed.no2"),2).cast("double"))
    .withColumn("conc_o3", round(col("components_parsed.o3"),2).cast("double"))
    .withColumn("conc_so2", round(col("components_parsed.so2"),2).cast("double"))
    .withColumn("conc_pm2_5", round(col("components_parsed.pm2_5"),2).cast("double"))
    .withColumn("conc_pm10", round(col("components_parsed.pm10"),2).cast("double"))
    .withColumn("conc_nh3", round(col("components_parsed.nh3"),2).cast("double"))
    # Derivar fecha desde dt
    .withColumn("fecha_ts", to_timestamp(col("dt").cast("long")))
    .withColumn("anio", year(col("fecha_ts")))
    .withColumn("mes", month(col("fecha_ts")))
    .withColumn("dia", dayofmonth(col("fecha_ts")))
    .withColumn("fecha", to_date(col("fecha_ts")))
    # Renombrar campos de ubicación
    .withColumnRenamed("ubigeo", "id_ubigeo")
    .withColumnRenamed("distrito", "nom_distrito")
    .withColumnRenamed("provincia", "nom_provincia")
    .withColumnRenamed("departamento", "nom_departamento")
    # Castear lat/long
    .withColumn("latitud", col("latitud").cast("double"))
    .withColumn("longitud", col("longitud").cast("double"))
    # fecha de actualizacion
    .withColumn("fecha_data", lit(current_date()))
    .drop("main_parsed", "components_parsed","main","components","dt")
)


In [0]:
# -------------------------------
# Escritura en Silver
# -------------------------------
# incremental (solo particiones afectadas)
if tipo_carga == "incremental":
    write_silver_air_pollution(df_silver, mode="incremental")
    
elif tipo_carga == "total":
    # total (reescribe toda la tabla)
    write_silver_air_pollution(df_silver, mode="total")


In [0]:
%sql
DELETE FROM silver.hd_lima_air_pollution where fecha  = '2026-03-29'

In [0]:
%sql
select * from silver.hd_lima_air_pollution where fecha  = '2026-03-29'
